In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression, LogisticRegression 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, r2_score,
roc_auc_score, roc_curve, mean_squared_error, mean_absolute_error, root_mean_squared_error)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier 
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

C:\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df_train = pd.read_csv("train_final.csv")

In [3]:
df

NameError: name 'df' is not defined

In [128]:
df_test = pd.read_csv("test_final.csv")

In [ ]:
dup_cols = df_train.columns[df_train.columns.duplicated()]
dup_cols

In [ ]:
dup_cols = df_test.columns[df_test.columns.duplicated()]
dup_cols

In [ ]:
df_train.columns

In [ ]:
cols = ['LATE_PAYMENT_MEAN', 'LATE_PAYMENT_MEAN.1']

# 1. Basic stats
#print("DESCRIBE:\n", df_train[cols].describe())

# 2. Null values
#print("\nNULLS:\n", df_train[cols].isnull().sum())

# 3. Zero values
#print("\nZEROS:\n", (df_train[cols] == 0).sum())

# 4. Correlation with TARGET
print("\nCORRELATION:\n", df_train[cols + ['TARGET']].corr())

In [129]:
df_train.drop(columns=['LATE_PAYMENT_MEAN'], inplace=True)
df_test.drop(columns=['LATE_PAYMENT_MEAN'], inplace=True)

In [130]:
df_train.rename(columns={'LATE_PAYMENT_MEAN.1': 'LATE_PAYMENT_MEAN'}, inplace=True)
df_test.rename(columns={'LATE_PAYMENT_MEAN.1': 'LATE_PAYMENT_MEAN'}, inplace=True)

In [131]:
# Replace inf → NaN in FULL dataframe
df_train = df_train.replace([np.inf, -np.inf], np.nan)

# Fill all NaN with 0 (or median if you want)
df_train = df_train.fillna(0)

In [85]:
X = df_train.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df_train['TARGET']

In [135]:
X = pd.get_dummies(X)

In [88]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [126]:
X_train.columns = X_train.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)
X_val.columns = X_val.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)

In [90]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [91]:
model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

In [92]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.283372 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17573
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 220
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.03
,n_estimators,800
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


In [94]:
y_prob = model.predict_proba(X_val)[:,1]

In [95]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_val, y_prob)
print("AUC-PR:", ap)

AUC-PR: 0.2784020314403072


In [96]:
import numpy as np
from sklearn.metrics import f1_score

best_f1 = 0
best_t = 0

for t in np.linspace(0.1, 0.9, 30):
    y_pred = (y_prob > t).astype(int)
    f1 = f1_score(y_val, y_pred)
    
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", best_t)
print("Best F1:", best_f1)

Best Threshold: 0.6517241379310346
Best F1: 0.33812154696132596


In [97]:
from sklearn.metrics import classification_report

y_final = (y_prob > best_t).astype(int)

print(classification_report(y_val, y_final))

              precision    recall  f1-score   support

           0       0.95      0.89      0.92     56538
           1       0.27      0.46      0.34      4965

    accuracy                           0.85     61503
   macro avg       0.61      0.68      0.63     61503
weighted avg       0.89      0.85      0.87     61503



In [106]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=0.3, random_state=42)

In [107]:
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

In [112]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [132]:
importance = pd.Series(model.feature_importances_, index=X.columns)
top_features = importance.nlargest(50).index 

In [116]:
# Filter your data to top features
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

# Undersample the majority class
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy={0: 56000, 1: 19000}, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train_top, y_train)

# verify
import collections
print(collections.Counter(y_resampled))

# Train LightGBM on the cleaned, undersampled data
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score

model_final = lgb.LGBMClassifier(
    n_estimators=500,
    scale_pos_weight=3,   # since we're at ~2:1 after undersampling, keep this light
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model_final.fit(X_resampled, y_resampled)

# Predict
y_pred = model_final.predict(X_test_top)
y_proba = model_final.predict_proba(X_test_top)[:, 1]

# Results
print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_proba))

Counter({0: 56000, 1: 19000})
[LightGBM] [Info] Number of positive: 19000, number of negative: 56000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058822 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11770
[LightGBM] [Info] Number of data points in the train set: 75000, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.253333 -> initscore=-1.080913
[LightGBM] [Info] Start training from score -1.080913
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56538
           1       0.18      0.68      0.29      4965

    accuracy                           0.73     61503
   macro avg       0.57      0.71      0.56     61503
weighted avg       0.90      0.73      0.79     61503

AUC-ROC: 0.772297486416376


In [117]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(y_test, (y_proba >= t).astype(int), pos_label=1) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
print("Best threshold:", best_threshold)

y_pred_best = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_best))

Best threshold: 0.6500000000000001
              precision    recall  f1-score   support

           0       0.95      0.88      0.91     56538
           1       0.25      0.47      0.33      4965

    accuracy                           0.84     61503
   macro avg       0.60      0.67      0.62     61503
weighted avg       0.89      0.84      0.86     61503



In [118]:
# Try a fixed higher threshold
y_pred_high = (y_proba >= 0.75).astype(int)
print(classification_report(y_test, y_pred_high))

              precision    recall  f1-score   support

           0       0.94      0.95      0.94     56538
           1       0.32      0.28      0.30      4965

    accuracy                           0.89     61503
   macro avg       0.63      0.62      0.62     61503
weighted avg       0.89      0.89      0.89     61503



In [120]:
from sklearn.metrics import precision_score, recall_score
# Fine-grained search between 0.65 and 0.75
thresholds = np.arange(0.65, 0.76, 0.01)
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, y_pred_t, pos_label=1)
    r = recall_score(y_test, y_pred_t, pos_label=1)
    f1 = f1_score(y_test, y_pred_t, pos_label=1)
    print(f"Threshold {t:.2f} → Precision: {p:.2f} | Recall: {r:.2f} | F1: {f1:.2f}")

Threshold 0.65 → Precision: 0.25 | Recall: 0.47 | F1: 0.33
Threshold 0.66 → Precision: 0.25 | Recall: 0.45 | F1: 0.32
Threshold 0.67 → Precision: 0.26 | Recall: 0.43 | F1: 0.32
Threshold 0.68 → Precision: 0.27 | Recall: 0.42 | F1: 0.32
Threshold 0.69 → Precision: 0.27 | Recall: 0.40 | F1: 0.32
Threshold 0.70 → Precision: 0.28 | Recall: 0.38 | F1: 0.32
Threshold 0.71 → Precision: 0.29 | Recall: 0.36 | F1: 0.32
Threshold 0.72 → Precision: 0.29 | Recall: 0.34 | F1: 0.32
Threshold 0.73 → Precision: 0.30 | Recall: 0.32 | F1: 0.31
Threshold 0.74 → Precision: 0.31 | Recall: 0.30 | F1: 0.31
Threshold 0.75 → Precision: 0.32 | Recall: 0.28 | F1: 0.30


In [139]:
df_train = pd.get_dummies(df_train, drop_first=True)

X_full = df_train.drop('TARGET', axis=1)
y_full = df_train['TARGET']

rus = RandomUnderSampler(sampling_strategy={0: 38000, 1: 19000}, random_state=42)
X_rus, y_rus = rus.fit_resample(X_full, y_full)

print("After undersampling:", collections.Counter(y_rus))
# Should print ~57k class 0, ~19k class 1

# Step 2 — Now split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_rus, y_rus, test_size=0.2, random_state=42, stratify=y_rus
)

X_train.columns = X_train.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)
X_val.columns = X_val.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)

# Step 3 — Train a fresh model on top features
# (reuse top_features from before OR recompute below)
temp_model = lgb.LGBMClassifier(n_estimators=200, random_state=42)
temp_model.fit(X_train, y_train)

importance = pd.Series(temp_model.feature_importances_, index=X_train.columns)
top_features = importance.nlargest(50).index

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

# Step 4 — Final model
model_final = lgb.LGBMClassifier(
    n_estimators=500,
    scale_pos_weight=2,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
model_final.fit(X_train_top, y_train)

# Step 5 — Evaluate
y_pred = model_final.predict(X_test_top)
y_proba = model_final.predict_proba(X_test_top)[:, 1]

print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_proba))

After undersampling: Counter({0: 38000, 1: 19000})
[LightGBM] [Info] Number of positive: 15200, number of negative: 30400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.107981 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17296
[LightGBM] [Info] Number of data points in the train set: 45600, number of used features: 204
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Number of positive: 15200, number of negative: 30400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029993 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11193
[LightGBM] [Info] Number of data points in the train set: 45600, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [

In [140]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(y_test, (y_proba >= t).astype(int), pos_label=1) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
print("Best threshold:", best_threshold)

y_pred_best = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_best))

Best threshold: 0.45000000000000007
              precision    recall  f1-score   support

           0       0.83      0.67      0.74      7600
           1       0.52      0.73      0.61      3800

    accuracy                           0.69     11400
   macro avg       0.68      0.70      0.68     11400
weighted avg       0.73      0.69      0.70     11400



In [7]:
df_0 = df_train[df_train['TARGET'] == 0]
df_1 = df_train[df_train['TARGET'] == 1]

df_0 = df_0.dropna(thresh=int(0.4 * df_train.shape[1]))

df_train = pd.concat([df_0, df_1])
#df_train = df_train.sample(frac=1, random_state=42)

In [ ]:
df_train.isnull().sum()

In [56]:
X_train.columns = X_train.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)
X_val.columns = X_val.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)

In [13]:
!pip install imblearn

  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)

   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 0/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imblearn]



In [14]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [30]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train_sm, y_train_sm)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [31]:
y_prob = model.predict_proba(X_val)[:,1]

In [32]:
for t in [0.5, 0.55, 0.6]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.55      0.03      0.06      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.51     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.55
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.62      0.02      0.04      4965

    accuracy                           0.92     61503
   macro avg       0.77      0.51      0.50     61503
weighted avg       0.90      0.92      0.88     61503


Threshold: 0.6
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.68      0.01      0.02      4965

    accuracy                           0.92     61503
   macro avg       0.80      0.51      0.49     61503
weighted avg       0.90  

In [57]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [58]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

In [59]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.498014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17575
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 221
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [60]:
for t in [0.5, 0.45, 0.4]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.55      0.03      0.06      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.51     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.45
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.52      0.05      0.10      4965

    accuracy                           0.92     61503
   macro avg       0.72      0.52      0.53     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.50      0.08      0.14      4965

    accuracy                           0.92     61503
   macro avg       0.71      0.54      0.55     61503
weighted avg       0.89  

In [61]:
df_train.shape

(307511, 116)

In [62]:
model = LGBMClassifier(
    n_estimators=1000,   # reduced
    learning_rate=0.02,
    num_leaves=34,
    max_depth=8,
    subsample=0.87,
    colsample_bytree=0.94,
    reg_alpha=0.04,
    reg_lambda=0.07,
    min_child_weight=39,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        # early stopping (new syntax)
        __import__('lightgbm').early_stopping(100)
    ]
)

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.527034 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17575
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 221
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.735808	valid_0's binary_logloss: 0.275768


,boosting_type,'gbdt'
,num_leaves,34
,max_depth,8
,learning_rate,0.02
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,39
,min_child_samples,20


In [63]:
for t in [0.5, 0.45, 0.4]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.55      0.03      0.06      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.51     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.45
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.52      0.05      0.10      4965

    accuracy                           0.92     61503
   macro avg       0.72      0.52      0.53     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.50      0.08      0.14      4965

    accuracy                           0.92     61503
   macro avg       0.71      0.54      0.55     61503
weighted avg       0.89  

In [ ]:
print(np.isinf(X_train).sum().sum())

In [64]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=20,   # smaller = less overprediction
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [65]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.477888 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17575
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 221
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,6
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [66]:
for t in [0.55, 0.6, 0.65, 0.7]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.55
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.62      0.02      0.04      4965

    accuracy                           0.92     61503
   macro avg       0.77      0.51      0.50     61503
weighted avg       0.90      0.92      0.88     61503


Threshold: 0.6
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.68      0.01      0.02      4965

    accuracy                           0.92     61503
   macro avg       0.80      0.51      0.49     61503
weighted avg       0.90      0.92      0.88     61503


Threshold: 0.65
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.71      0.01      0.01      4965

    accuracy                           0.92     61503
   macro avg       0.82      0.50      0.48     61503
weighted avg       0.90 

In [69]:
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

In [70]:
low_imp = feat_imp[feat_imp['importance'] < 20]['feature']

In [79]:
feat_imp.head(50)

,feature,importance
101,AMT_PAYMENT_SUM,334
44,EXT_SOURCE_MEAN,252
3,AMT_ANNUITY,243
58,CNT_PAYMENT_MEAN,238
78,DAYS_CREDIT_MAX,233
42,EXT_SOURCE_SUM,230
94,NUM_INSTALMENT_NUMBER_MAX,223
4,AMT_GOODS_PRICE,197
6,DAYS_BIRTH,193
62,APP_CREDIT_RATIO_MEAN,193


In [71]:
X_trainIF = X_train.drop(columns=low_imp)
X_valIF = X_val.drop(columns=low_imp)

In [73]:
for t in [0.4, 0.45, 0.5]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.50      0.08      0.14      4965

    accuracy                           0.92     61503
   macro avg       0.71      0.54      0.55     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.45
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.52      0.05      0.10      4965

    accuracy                           0.92     61503
   macro avg       0.72      0.52      0.53     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.55      0.03      0.06      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.51     61503
weighted avg       0.89  

In [74]:
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=25,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [75]:
model.fit(X_trainIF, y_train)

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.348996 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15137
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 88
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,boosting_type,'gbdt'
,num_leaves,25
,max_depth,6
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [76]:
for t in [0.4, 0.45, 0.5]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.92      0.99      0.96     56538
           1       0.50      0.08      0.14      4965

    accuracy                           0.92     61503
   macro avg       0.71      0.54      0.55     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.45
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.52      0.05      0.10      4965

    accuracy                           0.92     61503
   macro avg       0.72      0.52      0.53     61503
weighted avg       0.89      0.92      0.89     61503


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.55      0.03      0.06      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.51     61503
weighted avg       0.89  

# Step 1: replace inf → NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)

# Step 2: fill NaN
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)

In [ ]:
print(np.isinf(X_train).sum().sum())   # should be 0
print(X_train.isnull().sum().sum())    # should be 0

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=10,   # VERY IMPORTANT (imbalance)
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)

In [ ]:
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

In [ ]:
y_prob = model.predict_proba(X_val)[:,1]

for t in [0.5, 0.45, 0.4]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, y_pred))

In [ ]:
print(type(df_train))

In [38]:
X = df_train.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df_train['TARGET']

In [39]:
X_test = df_test.drop(['SK_ID_CURR'], axis=1)

In [40]:
X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

corr_target = df_train.select_dtypes(include=['number']).corr()['TARGET'].sort_values(ascending=False)
corr_target.head(21)

top_corr = corr_target[1:21]  # skip TARGET itself

plt.figure(figsize=(8,6))
sns.barplot(x=top_corr.values, y=top_corr.index)
plt.title("Top Features Correlated with TARGET")
plt.show()

In [ ]:
X_test = X_test[X.columns]

In [43]:
X.columns = X.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)
X_test.columns = X_test.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [ ]:
model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=40,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:, 1]

In [ ]:
feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(feat_imp)
feat_imp.to_csv('feature_importance.csv', index=False)

In [ ]:
zero_imp = feat_imp[feat_imp['importance'] == 0]['feature']

X = X.drop(columns=zero_imp)
X_test = X_test.drop(columns=zero_imp)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
LG = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

# train
LG.fit(X_train, y_train)

# predict
val_preds = LG.predict_proba(X_val)[:, 1]

# evaluate
auc = roc_auc_score(y_val, val_preds)
print("Validation AUC:", auc)

In [ ]:
test_preds = LG.predict_proba(X_test)[:, 1]

In [ ]:
# convert probabilities → 0/1
val_preds_binary = (val_preds > 0.5).astype(int)

# accuracy
acc = accuracy_score(y_val, val_preds_binary)
print("Accuracy:", acc)

# classification report
print(classification_report(y_val, val_preds_binary))

In [ ]:
for t in [0.5, 0.4, 0.3, 0.2]:
    preds = (val_preds > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, preds))

In [ ]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

LG1 = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

In [ ]:
LG1.fit(X_train, y_train)

val_preds1 = LG1.predict_proba(X_val)[:, 1]

In [ ]:
val_preds_binary1 = (val_preds1 > 0.3).astype(int)

In [ ]:
acc1 = accuracy_score(y_val, val_preds_binary1)
print("Accuracy:", acc1)

# classification report
print(classification_report(y_val, val_preds_binary1))

In [ ]:
for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    preds = (val_preds1 > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_val, preds))

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [ ]:
model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=40,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [ ]:
X.columns = X.columns.str.replace('[^A-Za-z0-9_]+', '_', regex=True)
X_test.columns = X_test.columns.str.replace('[^A-Za-z0-9_]+', '_', regex=True)

In [ ]:
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:, 1]

In [ ]:
val_preds_binary = (val_preds > 0.5).astype(int)

In [ ]:
acc2 = accuracy_score(y_val, val_preds_binary)
print("Accuracy:", acc2)

# classification report
print(classification_report(y_val, val_preds_binary))

In [ ]:
df_train['TARGET'].value_counts()

NameError: name 'pd' is not defined